# Configuration

In [ ]:
library(here)
library(tidyverse)
library(arrow)
library(skimr)
library(scales)
library(car)

source(here("src", "data_wrangling", "data_analysis.R"))
data_dir <- here("data", "intermediate", "whole_clinical_data.parquet")
output_dir <- here("results", "eda/")

# Load the data

In [ ]:
clinical_data <- read_parquet(data_dir)
head(clinical_data)

# Global summary
Let's check the column's data types and whether there are many missing values:

In [ ]:
skim(clinical_data) |> summary()

There are $720$ rows and $36$ columns: $22$ categorical (the target class can be found among these) and $14$ numeric features. Let's check each type separately.

## Numeric features
Let's sum up the data:

In [ ]:
# Numeric features summary
num_features_summary <- clinical_data |> 
  select(-code) |> 
  
  # Compute summary statistics for numeric features
  skim() |> yank("numeric") |> 
  
  select(c(skim_variable, mean, sd, complete_rate)) |> 
  rename(feature = skim_variable) |> 
  
  # Replace the rate of complete values with the rate of missing values
  mutate(
    missing_rate = 1 - complete_rate,
    complete_rate = NULL
    )

print(num_features_summary, digits = 4)

# Save results
output_file <- here(output_dir, "num_features_summary.csv")

num_features_summary |> 
  write.csv(output_file, row.names = FALSE)

We will drop the features with $missing \ rate > 25 \%$:

In [ ]:
num_features_summary |> 
  filter(missing_rate > 0.25) |> 
  select(c(feature, missing_rate))

Therefore, we will drop these features:

In [ ]:
clinical_data <- clinical_data |>
  select(-c(LAD, LAA, LAV))

## Categorical features
Let's sum up the data:

In [ ]:
# Categorical features summary
cat_features_summary <- clinical_data |> 
  select(-code) |> 
  
  # Compute summary statistics for categorical features
  skim() |> 
  yank("factor") |> 
  select(-c(complete_rate, ordered, top_counts)) |> 
  rename(feature = skim_variable) |> 
  mutate(
    missing_rate = n_missing/nrow(clinical_data),
    n_missing = NULL
    )

print(cat_features_summary, digits = 4)

# Save results
output_file <- here(output_dir, "cat_features_summary.csv")

cat_features_summary |> 
  write.csv(output_file, row.names = FALSE)

Using the same criteria as for the numeric features, we will drop those categorical features with $missing \ rate > 25 \%$:

In [ ]:
cat_features_summary |> 
  filter(missing_rate > 0.25) |> 
  select(c(feature, missing_rate))

Since there are no categorical features with a $missing \ rate > 25 \%$, we will keep all of them.

# Missing values
Let's check the distribution of the missing values stratified by the target class:

In [ ]:
plot_stratified_missingness(clinical_data, output = output_dir)

There is no significant difference in the distribution of the missing values between the two classes, so we can assume that the data is missing completely at random (MCAR).

Furthermore, we will delete those records and features where $missing\_rate > 25 \%$:

In [ ]:
plot_row_missingness(clinical_data, output = output_dir)

Since there are no records with more than $5$ missing values ($missing \ rate < 1 \%$), we will keep all of them.

# Statistical analysis


## Numeric features

In [ ]:
clinical_data |> 
  select(-c(code, hatch_score)) |> 
  plot_global_numeric_distribution(output = output_dir)

Numeric features distribution stratified by the target class:

In [ ]:
clinical_data |> 
  select(-c(code, hatch_score)) |> 
  plot_stratified_numeric_distribution(
    target_var = "AF_recurrence", 
    output = output_dir
    )

Numeric features distribution stratified by the intervention feature:

In [ ]:
clinical_data |> 
  select(-c(code, hatch_score)) |> 
  plot_stratified_numeric_distribution(
    target_var = "intervention", 
    output = output_dir
    )

## Categorical features

Categorical features distribution:

In [ ]:
clinical_data |> 
  plot_global_categorical_distribution(output = output_dir)

Categorical features distribution stratified by the target class:

In [ ]:
clinical_data |> 
  plot_stratified_categorical_distribution(
    target_var = "AF_recurrence",
    output = output_dir
    )

Categorical features distribution stratified by the intervention variable:

In [ ]:
clinical_data |> 
  plot_stratified_categorical_distribution(
    target_var = "intervention", 
    output = output_dir
    )

## Multicolinearity
Now, let's check the correlation between features.

### Numeric features
Correlation matrix of numeric features:

In [ ]:
clinical_data |> select(-code) |> 
  plot_corr_matrix(threshold = 0.5, output = output_dir)

Although the waist to height ratio might perform better as an obesity indicator (check the numeric distribution stratified by AF recurrence), the `BMI` feature's missing rate is $0$. Therefore, we will drop the `waist_height` feature:

In [ ]:
clinical_data <- clinical_data |> 
  select(-waist_height_ratio)


### Categorical features
To address categorical features' colinearity, we will use **Cramer's V**.

In [ ]:
clinical_data |> 
  plot_corr_matrix(dtype = "cat", threshold = 0.5, output = output_dir) |> 
  suppressWarnings()

There is a perfect correlation between `heart_failure`-`LVEF` and a significant correlation between `hypolipidemic_meds`-`hypercholesterolemia`. Since the latter ones are more informative, we will drop `heart_failure` and `hypolipidemic_meds` feature.

In [ ]:
clinical_data <- clinical_data |> 
  select(-c(heart_failure, hypolipidemic_meds))

### Global correlation
To address global colinearity, we will use the **Variance Inflation Factor**:

$$
VIF_i = \frac{1}{1 - R_i^2}
$$

Where $R_i^2$ is the determination coefficient of the regression of the $i$-th feature on the remaining features.

In [ ]:
# VIF calculation
clinical_data |> 
select(-code) |>
plot_vif(output = output_dir)

Since $VIF < 5$ for all of the features, there is no more multicollinearity.

# Saving the data

In [ ]:
# Full data set
write_parquet(
    clinical_data, 
    "data/clean/whole_clinical_data.parquet"
    )

head(clinical_data)